# 01 — HMM on an IBL visual→higher-order→motor set (Phase 1 + Phase 2)

Interactive exploration, in four parts:

1. **Download & load** one session that records our regions simultaneously.
2. **Look at the spikes** — firing-rate table, rasters, PSTHs.
3. **Fit a Poisson HMM** per region (pre-stimulus baseline biases the initial state).
4. **Phase 2: cross-validate K** (number of states) by held-out log-likelihood.

**Regions.** `VISp` (primary visual) → `VISpm` (higher-order visual / integration) → `SCm`, `MRN` (motor midbrain). This is the closest IBL comes to a *visual → higher-order → motor* hierarchy **in one session** — see the closing note on why a cleaner hierarchy needs SHIELD.

Everything heavy lives in `src/`; this notebook orchestrates and plots. Colours use the **Okabe-Ito** palette (red-green-colour-blind safe) and **cividis** heatmaps. Figures save under `results/figures/`. Run top-to-bottom with the `neurodatarehack` kernel.

## Part 0 — Setup & parameters

Speed knobs: `MAX_TRIALS` subsamples trials (HMM cost scales with trials×bins); `N_ITER` caps EM iterations. Long cells print their runtime so you can extrapolate when scaling up.

In [ ]:
import sys, time, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from src.data.access import get_one, sessions_table, load_trials, load_units
from src.data.build_matrices import (
    region_tensor_from_probes,   # bin a region's spikes -> (trials, time, neurons)
    region_event_spikes,         # raw in-window spike times, for raster plots
    firing_rate_table,           # per-unit firing statistics
    drop_silent_units,
)
from src.dynamics.hmm import (
    fit_poisson_hmm,             # fit + baseline-biased startprob + state ordering
    cross_validate_states,       # Phase 2: choose K by held-out log-likelihood
    net_flux, baseline_exit_latency, forward_fraction_null,
)
from src.viz.style import set_style, state_colors, region_color, savefig, SEQUENTIAL, DIVERGING

set_style()   # apply the colour-vision-safe palette + clean defaults

# ---------------------- parameters ----------------------
PARAMS = yaml.safe_load((ROOT / "config" / "params.yaml").read_text())
SESSION_EID = "5c0c560e-9e1f-45e9-b66e-e4ee7855be84"   # has VISp, VISpm, SCm, MRN; 843 trials
REGIONS = ["VISp", "VISpm", "SCm", "MRN"]              # sensory -> higher-order -> motor

PRE_S, POST_S, BIN_S = 0.5, 0.8, 0.02                  # 0.5 s pre-stimulus baseline; 20 ms bins
N_STATES = 5          # provisional; Part 4 cross-validates this
N_ITER   = 30
MAX_TRIALS = 400      # subsample for speed; set None for all ~843
SEED = 0

# ONE cache lives inside the repo (data/ is gitignored)
CACHE = str((ROOT / PARAMS["data"]["one_cache_dir"]).resolve())
rng = np.random.default_rng(SEED)
print("regions:", REGIONS, "| cache:", CACHE)

## Part 1 — Download & load

Data is **downloaded and cached** (not streamed) into `data/ONE/` inside the repo. First run of a session pulls ~440 MB (both probes, good units only); re-runs are instant.

In [ ]:
t0 = time.time()
one = get_one(cache_dir=CACHE)
pids = sessions_table(one).query("eid == @SESSION_EID")["pid"].tolist()
print(f"session {SESSION_EID}\n{len(pids)} probes: {[p[:8] for p in pids]}   [{time.time()-t0:.1f}s]")

In [ ]:
# Trials + the paper's inclusion mask, then optionally subsample for speed.
trials, mask = load_trials(one, SESSION_EID)
good = trials[mask].reset_index(drop=True)
if MAX_TRIALS is not None and len(good) > MAX_TRIALS:
    good = good.iloc[np.sort(rng.choice(len(good), MAX_TRIALS, replace=False))].reset_index(drop=True)

events = good["stimOn_times"].values
rt = (good["firstMovement_times"] - good["stimOn_times"]).values   # reaction time (s)
print(f"using {len(good)} trials (of {mask.sum()} usable)  |  median RT = {np.nanmedian(rt)*1000:.0f} ms")

In [ ]:
# Load good units for both probes; tag each with its Beryl region.
# clusters['acronym'] is the fine Allen label (e.g. SCig); Beryl is the rollup we filter on.
t0 = time.time()
from iblatlas.regions import BrainRegions
br = BrainRegions()

probe_loads = []
for pid in pids:
    spikes, clusters = load_units(one, pid)
    clusters = clusters.copy()
    clusters["Beryl"] = br.id2acronym(clusters["atlas_id"].values, mapping="Beryl")
    probe_loads.append({"spikes": spikes, "clusters": clusters})
    print(f"probe {pid[:8]}: {clusters.shape[0]} good units")
print(f"[{time.time()-t0:.1f}s]")

In [ ]:
# Build, once, the binned tensor and the raw raster spikes per region.
tensors, rasters = {}, {}
for region in REGIONS:
    counts, bin_centers = region_tensor_from_probes(
        probe_loads, region, events, pre_s=PRE_S, post_s=POST_S, bin_size_s=BIN_S)
    counts, _ = drop_silent_units(counts, min_total_spikes=1)   # drop units silent in-window
    tensors[region] = (counts, bin_centers)
    rasters[region] = region_event_spikes(probe_loads, region, events, pre_s=PRE_S, post_s=POST_S)
    print(f"{region}: tensor {counts.shape}  (trials, bins, units)")

## Part 2 — Look at the spikes

### 2a. Firing statistics

The units are IBL's **good** units (`qc == 1.0`: they pass every IBL quality metric — refractory period, amplitude, presence, etc.). "Good" means *well-isolated*, **not** *highly active or task-tuned*. Many good units fire slowly, which is why a single-trial raster can look sparse. The table quantifies that: `median_frac_active` near 0 and a large `n_units_below_1hz` flag mostly-silent populations. If we later want livelier units we can pre-filter on these stats (or use MSR), but for an unbiased first look we keep all good units.

In [ ]:
rows = []
for region in REGIONS:
    counts, _ = tensors[region]
    ft = firing_rate_table(counts, BIN_S)
    rows.append({
        "region": region,
        "n_units": counts.shape[2],
        "pop_mean_rate_hz": round(ft.mean_rate_hz.mean(), 2),
        "median_unit_rate_hz": round(ft.mean_rate_hz.median(), 2),
        "median_spikes_per_trial": round(ft.spikes_per_trial.median(), 2),
        "median_frac_active": round(ft.frac_trials_active.median(), 2),
        "n_units_below_1hz": int((ft.mean_rate_hz < 1).sum()),
    })
print("Population firing summary (per region):")
display(pd.DataFrame(rows))

print("\nPer-unit firing stats — VISp (top and bottom 5 by rate):")
ft_visp = firing_rate_table(tensors["VISp"][0], BIN_S)
display(pd.concat([ft_visp.head(5), ft_visp.tail(5)]))

### 2b. Rasters & PSTHs

- **Raster** (one trial): each mark is a spike, rows are units. Look for a change at `t = 0`.
- **Population PSTH**: mean rate across units & trials (± SEM across trials).
- **Per-unit PSTH heatmap** (cividis): each row a unit, peak-normalised, sorted by peak time.

In [ ]:
def plot_region_activity(region, example_trial=0):
    """Single-trial spike raster, population PSTH, and per-unit PSTH heatmap."""
    counts, bc = tensors[region]
    trial_idx, unit_idx, t_rel = rasters[region]
    n_units = counts.shape[2]
    col = region_color(region)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={"width_ratios": [1, 1, 1.1]})

    # (1) dot raster for one example trial
    m = trial_idx == example_trial
    ax[0].scatter(t_rel[m], unit_idx[m], s=5, c="k", marker="|", linewidths=0.6)
    ax[0].axvline(0, color=col, ls="--", lw=1.2)
    ax[0].set(xlabel="time from stimOn (s)", ylabel="unit",
              title=f"{region}: spike raster (trial {example_trial})", xlim=(bc[0], bc[-1]))

    # (2) population PSTH (mean rate across units & trials, SEM across trials)
    psth = counts.mean(axis=(0, 2)) / BIN_S
    sem = counts.mean(axis=2).std(axis=0) / np.sqrt(counts.shape[0]) / BIN_S
    ax[1].plot(bc, psth, color=col)
    ax[1].fill_between(bc, psth - sem, psth + sem, alpha=0.3, color=col)
    ax[1].axvline(0, color="0.3", ls="--", lw=1)
    ax[1].set(xlabel="time from stimOn (s)", ylabel="rate (sp/s)",
              title=f"{region}: population PSTH (n={n_units})")

    # (3) per-unit PSTH heatmap, peak-normalised, sorted by peak latency (cividis)
    unit_psth = counts.mean(axis=0).T / BIN_S
    norm = unit_psth / (unit_psth.max(axis=1, keepdims=True) + 1e-9)
    order = np.argsort(norm.argmax(axis=1))
    im = ax[2].imshow(norm[order], aspect="auto", cmap=SEQUENTIAL,
                      extent=[bc[0], bc[-1], n_units, 0])
    ax[2].axvline(0, color="w", ls="--", lw=1)
    ax[2].set(xlabel="time from stimOn (s)", ylabel="unit (sorted by peak time)",
              title=f"{region}: per-unit PSTH (peak-norm)")
    fig.colorbar(im, ax=ax[2], label="normalised rate")
    fig.tight_layout()
    savefig(fig, "phase2_activity", f"{region}_activity", root=ROOT)
    return fig

for region in REGIONS:
    plot_region_activity(region)
plt.show()

## Part 3 — Poisson HMM per region

`fit_poisson_hmm` (see `src/dynamics/hmm.py`):
1. fits a Poisson HMM on **raw counts** (Poisson emissions need integers — no smoothing);
2. **biases the initial state** — installs the pre-stimulus baseline occupancy as `startprob_` (details below);
3. **orders states by occupancy time**, so state 0 is baseline/earliest.

### What "bias with startprob" means

An HMM has three learnable pieces: emissions (each state's firing pattern), the transition matrix, and **`startprob`** = the distribution over which state a sequence *begins* in. By default EM learns `startprob` from the first bin of each trial. We instead set it to the **average state occupancy across the whole pre-stimulus baseline**, so Viterbi decoding is anchored: every trial is assumed to start in the baseline regime, and the post-stimulus path is read as a *departure from baseline*. Turn it off with `bias_startprob=False` to compare.

### Reading the three panels (your questions, answered)

- **Left — state raster.** Colour = the most-likely (Viterbi) **state index** at each (trial, time), on the cividis scale (see colourbar). Each row is one trial; rows are **sorted by reaction time**. The white curve is each trial's **first-movement time**. The window starts at −0.5 s because that is our pre-stimulus **baseline**; the dashed line is stimulus onset (0 s).
- **Middle — occupancy.** For each state *k*, the line is P(state = *k* | time) = the **fraction of trials** in state *k* at that time bin. States are discrete *per trial*, but *averaged across the 400 trials* the occupancy is a smooth fraction in [0, 1]. The shaded band is the baseline.
- **Right — net flux.** See the markdown after the plot for the exact `forward_fraction` formula.

In [ ]:
t0 = time.time()
results = {}
for region in REGIONS:
    counts, bc = tensors[region]
    res = fit_poisson_hmm(counts, N_STATES, bc < 0.0, n_iter=N_ITER, seed=SEED, bias_startprob=True)
    res.bin_centers = bc
    res.latency = baseline_exit_latency(res.occupancy, bc, baseline_state=0, thresh=0.5)
    results[region] = res
    lat = "nan" if np.isnan(res.latency) else f"{res.latency*1000:.0f} ms"
    print(f"{region}: converged={res.model.monitor_.converged}  "
          f"forward_fraction={res.forward_fraction:.3f}  baseline_exit={lat}")
print(f"[{time.time()-t0:.1f}s total for {len(REGIONS)} regions]")

In [ ]:
def plot_hmm(region, res):
    """State raster (trials sorted by RT), state occupancy, and net-flux matrix."""
    bc, K = res.bin_centers, res.occupancy.shape[0]
    cols = state_colors(K)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

    # (1) most-likely state per (trial, time); trials sorted by reaction time
    order = np.argsort(rt)
    im0 = ax[0].imshow(res.states[order], aspect="auto", interpolation="nearest",
                       extent=[bc[0], bc[-1], len(order), 0], cmap=SEQUENTIAL, vmin=0, vmax=K-1)
    ax[0].plot(rt[order], np.arange(len(order)) + 0.5, color="w", lw=1.2, label="first move")
    ax[0].axvline(0, color="w", ls="--", lw=1)
    ax[0].set(xlabel="time from stimOn (s)", ylabel="trial (sorted by RT)",
              title=f"{region}: HMM state (K={K})")
    ax[0].legend(frameon=False, loc="lower right", fontsize=8, labelcolor="w")
    fig.colorbar(im0, ax=ax[0], label="state index", ticks=range(K))

    # (2) occupancy P(state | time); state 0 = baseline (shaded pre-stimulus window)
    for k in range(K):
        ax[1].plot(bc, res.occupancy[k], color=cols[k], label=f"state {k}")
    ax[1].axvspan(bc[0], 0, color="0.9")
    ax[1].axvline(0, color="0.3", ls="--", lw=1)
    ax[1].set(xlabel="time from stimOn (s)", ylabel="P(state)", title="state occupancy")
    ax[1].legend(frameon=False, fontsize=8, ncol=2)

    # (3) net transition flux N - N^T. Red above diagonal = forward; blue = a return step.
    flux = net_flux(res.trans_counts)
    vmax = np.abs(flux).max() or 1
    im = ax[2].imshow(flux, cmap=DIVERGING, vmin=-vmax, vmax=vmax)
    for i in range(K):
        for j in range(K):
            if i != j:
                ax[2].text(j, i, f"{flux[i,j]:+d}", ha="center", va="center", fontsize=8)
    ax[2].set(xlabel="to state", ylabel="from state", xticks=range(K), yticks=range(K),
              title=f"net flux N−Nᵀ (fwd frac={res.forward_fraction:.2f})")
    fig.colorbar(im, ax=ax[2], label="net i→j transitions")
    fig.tight_layout()
    savefig(fig, "phase2_hmm", f"{region}_hmm", root=ROOT)
    return fig

for region in REGIONS:
    plot_hmm(region, results[region])
plt.show()

### How the net-flux matrix and `forward_fraction` are computed

From the Viterbi paths we count **observed transitions** `N[i,j]` = number of times a trial stepped from state *i* to state *j* between consecutive bins (within a trial). The **net flux** is the antisymmetric part

$$\Phi[i,j] = N[i,j] - N[j,i]$$

so $\Phi[i,j] > 0$ (red, above the diagonal) = *more i→j than j→i* = a net forward step; $\Phi[i,j] < 0$ (blue above the diagonal) = a net **return** step. `forward_fraction` collapses this to one number:

$$\text{forward\_fraction} = \frac{\sum_{j>i} N[i,j]}{\sum_{i \ne j} N[i,j]}$$

— the share of all off-diagonal transitions running to a *later-ordered* state. Because states are ordered by their occupancy time, this measures **directedness**, not the absolute arrow of time: **0.5 = reversible mixing**, **→1 = a clean directed metastable chain**. Validated on synthetic forward/reversed/shuffled data (shuffled → 0.5).

> **Earlier run (yours), with the shuffle null:** VISp 0.55, SCm 0.83, MRN 0.82 — all **p < 0.001** vs a null of 0.50 ± 0.002. So even VISp is *significantly* directed, just far weaker than the two motor areas.

> **On your SCm↔MRN feedback idea.** The picture you describe — one area's state rising as another's falls, MRN peaking *between* two SCm peaks, hinting SCm→MRN→SCm before a shared terminal attractor — is exactly what to make quantitative. Within one area, `forward_fraction < 1` with **specific negative (blue) off-diagonal entries** is the signature of *return* transitions, i.e. a local feedback loop in the flux matrix. Across areas, the honest tools are (i) **cross-correlation of the two regions' state-occupancy time series** (does MRN's rise lag SCm's, and by how much?), and (ii) a **joint / multi-region HMM** (or rSLDS) whose states live on the *concatenated* SCm+MRN population, so one state can mean “both engaged” and transitions can encode the exchange. That joint-state model is the right Phase-3 vehicle for the triple-feedback hypothesis; the per-region HMMs here are the groundwork.

In [ ]:
# Cross-area directionality: order regions by (sustained) baseline exit.
print("Baseline-exit latency (inter-areal arrow; nan = no dominant baseline to leave):")
lat = sorted(((r, results[r].latency) for r in REGIONS), key=lambda x: (np.isnan(x[1]), x[1]))
print("  " + "  →  ".join(f"{r} ({'nan' if np.isnan(t) else f'{t*1000:.0f} ms'})" for r, t in lat))

## Part 4 — Phase 2: cross-validate the number of states K

K was fixed at 5 above. Here we choose it honestly: split **trials** into folds, fit on the training trials, and score the **held-out** trials. Held-out log-likelihood rises with K then plateaus (or dips) once extra states just overfit — the plateau onset is the defensible K.

**This is the slow cell.** Cost ≈ (#K) × folds × regions HMM fits. It subsamples harder (`CV_MAX_TRIALS`) and prints per-region timing so you can extrapolate.

In [ ]:
CV_KS = [2, 3, 4, 5, 6, 7]
CV_FOLDS = 3
CV_ITER = 25
CV_MAX_TRIALS = 250     # CV refits many times; subsample harder than the single fits above

cv_results = {}
for region in REGIONS:
    counts, bc = tensors[region]
    if CV_MAX_TRIALS and counts.shape[0] > CV_MAX_TRIALS:
        counts = counts[np.sort(rng.choice(counts.shape[0], CV_MAX_TRIALS, replace=False))]
    t0 = time.time()
    cv = cross_validate_states(counts, CV_KS, n_folds=CV_FOLDS, n_iter=CV_ITER, seed=SEED)
    cv_results[region] = cv
    print(f"{region}: best K = {cv.attrs['best_k']}   [{time.time()-t0:.0f}s, {len(CV_KS)*CV_FOLDS} fits]")

In [ ]:
# CV curves: held-out log-likelihood vs K, one line per region.
fig, ax = plt.subplots(figsize=(7, 4.5))
for region in REGIONS:
    cv = cv_results[region]
    ax.errorbar(cv.n_states, cv.mean_test_ll, yerr=cv.std_test_ll,
                marker="o", capsize=3, color=region_color(region), label=region)
    bk = cv.attrs["best_k"]
    ax.scatter([bk], [cv.loc[cv.n_states == bk, "mean_test_ll"].iloc[0]],
               s=140, facecolors="none", edgecolors=region_color(region), linewidths=2, zorder=5)
ax.set(xlabel="number of HMM states K", ylabel="held-out log-likelihood / sample",
       title="Phase 2: cross-validated K (circled = best per region)")
ax.legend(frameon=False)
fig.tight_layout()
savefig(fig, "phase2_cv", "cv_states", root=ROOT)
plt.show()
print("Best K per region:", {r: cv_results[r].attrs["best_k"] for r in REGIONS})

In [ ]:
# Refit each region at its cross-validated K; optionally test directedness vs a null.
RUN_NULL = False   # each shuffle is a full HMM fit; flip on for significance (slow)
N_SHUFFLES = 10

final = []
for region in REGIONS:
    counts, bc = tensors[region]
    K = cv_results[region].attrs["best_k"]
    res = fit_poisson_hmm(counts, K, bc < 0.0, n_iter=N_ITER, seed=SEED, bias_startprob=True)
    lat = baseline_exit_latency(res.occupancy, bc, thresh=0.5)
    row = {"region": region, "best_K": K, "forward_fraction": round(res.forward_fraction, 3),
           "baseline_exit_ms": None if np.isnan(lat) else round(lat*1000)}
    if RUN_NULL:
        t0 = time.time()
        null = forward_fraction_null(counts, K, bc < 0.0, n_shuffles=N_SHUFFLES, seed=SEED)
        row["null_mean"] = round(float(null.mean()), 3)
        row["p_vs_null"] = round(float((null >= res.forward_fraction).mean()), 3)
        print(f"{region}: null done [{time.time()-t0:.0f}s]")
    final.append(row)
final_df = pd.DataFrame(final)
display(final_df)
(ROOT / "results" / "hmm").mkdir(parents=True, exist_ok=True)
final_df.to_csv(ROOT / "results" / "hmm" / "phase2_final.csv", index=False)
print("saved -> results/hmm/phase2_final.csv")

## On the cross-area latency being fragile (your question 5)

The first pass reported `VISp (10 ms) → MRN (10 ms) → SCm (110 ms)`. Three problems made that untrustworthy, now fixed:

1. **A 10 ms “exit” is implausible.** Median RT is ~170 ms and even V1's visual latency is ~40–50 ms, so “leaves baseline at 10 ms” cannot be a real evoked departure.
2. **Why it happened.** The old metric took the *first single bin* where baseline occupancy dipped below 0.5. If a region's baseline state was only ~0.5 occupied to begin with (baseline not strongly dominant), ordinary bin-to-bin wobble crosses 0.5 almost immediately — a threshold artifact, not a transition.
3. **The fix.** `baseline_exit_latency` now (a) returns `nan` unless the baseline state is genuinely dominant pre-stimulus (mean occupancy > 0.5), and (b) requires the crossing to be **sustained** for ≥ 3 consecutive bins (60 ms). So a region reports a latency only if it *had* a real baseline and *durably* left it.

That is why `forward_fraction` — validated against forward/reversed/shuffled synthetic data — is the more trustworthy readout, and why cross-area *timing* should really use **state-occupancy cross-correlation** + a **joint multi-region model** (Part 3 note), not a single threshold.

## Caveats & next steps

- One session, subsampled trials. Set `MAX_TRIALS=None`, re-run, and repeat on the replicate sessions before believing any number.
- `forward_fraction` measures directedness, not absolute time direction (fixed by the state ordering). Your earlier `RUN_NULL` run showed all three regions significant (p<0.001).
- **Better hierarchy needs SHIELD.** A clean *visual → higher-order → motor* triplet (e.g. VISp → PL/ACA → MOs/MOp) is co-recorded in **one** SHIELD session with hundreds of units each; in IBL it essentially never co-occurs (VISp + higher-visual + motor: 2 sessions in the whole release). `VISpm` here is the best IBL stand-in for the “higher-order” tier.
- **The feedback hypothesis** (SCm↔MRN) wants a joint multi-region HMM / rSLDS and occupancy cross-correlation — the natural Phase 3 step.